# Open-Coding

Empirical half of taxonomy construction (Section 3.2 "Grounding in observation" / Appendix A.4):
`dump_random_traces` writes raw excerpts for the manually-coded stratified subset (paper: "48 traces");
`annotate` runs DeepSeek-assisted open coding on groups of 10 traces (paper: "800 additional traces").

In [ ]:
import os
import json
import random
from tqdm import tqdm
import pandas as pd

from src.datasets import get_or_create_dataset
from src.model_configurations import deepseek_reasoner
from src.prompt_util import prompt_deepseek

In [ ]:
eedi_dataset = get_or_create_dataset("eedi_data", n_limit=500)
sciq_dataset = get_or_create_dataset("sciq_data", n_limit=500)

### Shared helpers: LLM-based category induction

In [ ]:
def format_examples(df, n: int = 5):
    examples = df.sample(min(len(df), n))
    return [{
        "id": e["Id"],
        "problem": e["problem"],
        "reasoning": e["reasoning"]
    } for i,e in examples.iterrows()]

def discover_categories_from_traces(examples, subject: str):
    """
    examples: list of dicts like:
        [
            {"id": "1", "problem": "...", "reasoning": "..."},
            {"id": "2", "problem": "...", "reasoning": "..."},
            ...
        ]
    subject: e.g. "math" or "science"
    """
    
    # Build a single text block with the examples
    traces_text = []
    for ex in examples:
        traces_text.append(
            f"Trace {ex['id']}:\nProblem: {ex['problem']}\nReasoning:\n{ex['reasoning']}\n"
        )
    traces_text = "\n\n".join(traces_text)

    system_prompt = (
        "You are an expert in cognitive task analysis, think-aloud protocol analysis, "
        "and grounded theory coding. Your task is to derive inductive categories of cognitive "
        f"processes from reasoning traces of experts that are generating {subject} distractors for a multiple choice exam. Follow qualitative analysis best practices: "
        "bottom-up coding, constant comparison, memoing, and grounding all claims in the data."
    )

    user_prompt = f"""
I will provide you with a list of reasoning traces. Your task is to discover
common categories of reasoning or cognitive behaviors of the experts that are generating {subject} distractors, using a systematic inductive approach.

Your responsibilities:

1. Identify recurring cognitive behaviors or steps.
2. For every category, provide:
   - A clear definition (1-2 sentences)
   - A description of what behaviors fall under it
3. Provide 2-3 grounded citations for each category:
   - Verbatim excerpts from the traces
   - Include trace ID and step number (or text location)
4. Do not invent anything not present in the traces.
5. Focus only on recurring categories, not one-off behaviors.

OUTPUT FORMAT:

## Discovered Categories

### Category 1: [Name]
**Definition:**
[...]
**Example Citations:**
- "..." (Trace X, Step Y)
- "..." (Trace A, Step B)
- "..." (Trace C, Step D)

### Category 2: [Name]
**Definition:**
[...]

(Continue as needed.)

After listing categories, include:

## Notes on Method & Coverage
Explain how the categories were derived and how representative they are.

---

Here are the reasoning traces:

{traces_text}
"""

    return prompt_deepseek(system_prompt, user_prompt, deepseek_reasoner)

In [ ]:
# LLM-assisted open coding: DeepSeek induces categories over batches of 10 traces
# (paper's "800 additional traces", no previously extracted strategies provided)
def annotate(data_folder, dataset, responses, results_df, model: str, mode: str, subject: str, include_unsolvable: bool = True):
    reasoning_key = "step_by_step" if mode == "cot" else "raw_reasoning"
    datapoints = []
    for k,response in responses.items():
        dp = dataset[int(k)]
        try:
            result = results_df[results_df["Id"] == int(k)].iloc[0]
        except:
            print(f"Could not find {k} in {set(results_df['Id'])}")
        datapoints.append((k, dp["Problem"]["Question"], response[reasoning_key], result["proportional_match"], result["number_correct"], result["distractors"], dp["Problem"]["Solvable"], result["num_cor_sol_steps_by_datapointid"]))
    df = pd.DataFrame(datapoints, columns=["Id", "problem", "reasoning", "proportional_match", "number_correct", "distractors", "solvable", "nr_steps_of_solution"])

    out_dir = os.path.join(data_folder, "category_induction", mode, model)
    os.makedirs(out_dir, exist_ok=True)

    # stratification — matches the manual inspection buckets
    low_match_df = df[df["proportional_match"] < 0.5]
    high_match_df = df[df["proportional_match"] > 0.5]

    buckets = [
        ("low_match", low_match_df),
        ("high_match", high_match_df),
    ]
    if include_unsolvable:
        buckets.append(("unsolvable", df[~df["solvable"]]))

    for bucket_name, bucket_df in buckets:
        for i in tqdm(range(5), desc=f"{model}/{mode}/{bucket_name}"):
            out_path = os.path.join(out_dir, f"{bucket_name}_{i}.md")
            if os.path.exists(out_path):
                continue
            if len(bucket_df) == 0:
                continue
            reasoning, answer = discover_categories_from_traces(format_examples(bucket_df, 10), subject)
            with open(out_path, "w") as f:
                f.write(answer)

In [ ]:
JOINT_SETTINGS = {
    ("deepseek", "cot"): "deepseek-naive-cot-deepseek-chat",
    ("deepseek", "reasoning"): "deepseek-naive-deepseek-reasoner",
    ("glm", "cot"): "openrouter-naive-cot-z-ai_glm-4.7-chat",
    ("glm", "reasoning"): "openrouter-naive-z-ai_glm-4.7-reasoner",
}

# manual open coding: writes the raw trace excerpts a human annotator later codes by hand
# (paper's stratified subset of 48 traces)
def dump_random_traces(data_folder: str, dataset, n_per_bucket: int = 4, seed: int = 42):
    """
    Dump a stratified random sample of traces for manual inspection.
    Samples n_per_bucket traces from each of (model x mode x prop_match bucket) combinations,
    where model in {deepseek, glm}, mode in {cot, reasoning}, bucket in {low_match, high_match}
    split at 0.5. Total = 2 * 2 * 2 * n_per_bucket traces.
    Skips traces whose output file already exists.
    """
    out_dir = os.path.join("manual_inspection", "opencoding", "joint", data_folder)
    os.makedirs(out_dir, exist_ok=True)
    rng = random.Random(seed)

    for (model, mode), filename in JOINT_SETTINGS.items():
        reasoning_key = "step_by_step" if mode == "cot" else "raw_reasoning"
        responses_path = os.path.join(data_folder, "joint_results", f"{filename}_responses_by_datapointid.json")
        results_path = os.path.join(data_folder, "joint_results", f"{filename}_results.csv")
        with open(responses_path, "r") as f:
            responses = json.load(f)
        results_df = pd.read_csv(results_path)

        low_ids = results_df[results_df["proportional_match"] < 0.5]["Id"].tolist()
        high_ids = results_df[results_df["proportional_match"] > 0.5]["Id"].tolist()
        rng.shuffle(low_ids)
        rng.shuffle(high_ids)

        for bucket, ids in [("low_match", low_ids[:n_per_bucket]), ("high_match", high_ids[:n_per_bucket])]:
            for dpid in ids:
                path = os.path.join(out_dir, f"{model}_{mode}_{bucket}_{dpid}.txt")
                if os.path.exists(path):
                    continue
                dp = dataset[int(dpid)]
                resp = responses[str(dpid)]
                row = results_df[results_df["Id"] == dpid].iloc[0]
                content = (
                    f"Setting: {model} / {mode}\n"
                    f"Bucket: {bucket} (proportional_match split at 0.5)\n"
                    f"Id: {dpid}\n"
                    f"Proportional match: {row['proportional_match']}\n"
                    f"Number correct: {row['number_correct']}\n\n"
                    f"Question:\n{dp['Problem']['Question']}\n\n"
                    f"Correct answer: {dp['Choices']['CorrectAnswer']}\n"
                    f"Groundtruth distractors: {dp['Choices']['Distractors']}\n"
                    f"Model distractors: {row['distractors']}\n\n"
                    f"Reasoning:\n{resp[reasoning_key]}\n"
                )
                with open(path, "w") as fout:
                    fout.write(content)

    print(f"Wrote traces to {out_dir}")

## Eedi

In [ ]:
dump_random_traces("eedi_data", eedi_dataset, n_per_bucket=4)

### CoT

In [ ]:
with open("eedi_data/joint_results/deepseek-naive-cot-deepseek-chat_responses_by_datapointid.json", "r") as f:
    responses = json.load(f)
results_df = pd.read_csv("eedi_data/joint_results/deepseek-naive-cot-deepseek-chat_results.csv")

annotate("eedi_data", eedi_dataset, responses, results_df, "deepseek", "cot", "math")

In [ ]:
with open("eedi_data/joint_results/openrouter-naive-cot-z-ai_glm-4.7-chat_responses_by_datapointid.json", "r") as f:
    responses = json.load(f)
results_df = pd.read_csv("eedi_data/joint_results/openrouter-naive-cot-z-ai_glm-4.7-chat_results.csv")

annotate("eedi_data", eedi_dataset, responses, results_df, "glm", "cot", "math")

### Reasoning

In [ ]:
with open("eedi_data/joint_results/deepseek-naive-deepseek-reasoner_responses_by_datapointid.json", "r") as f:
    responses = json.load(f)
results_df = pd.read_csv("eedi_data/joint_results/deepseek-naive-deepseek-reasoner_results.csv")

annotate("eedi_data", eedi_dataset, responses, results_df, "deepseek", "reasoning", "math")

In [ ]:
with open("eedi_data/joint_results/openrouter-naive-z-ai_glm-4.7-reasoner_responses_by_datapointid.json", "r") as f:
    responses = json.load(f)
results_df = pd.read_csv("eedi_data/joint_results/openrouter-naive-z-ai_glm-4.7-reasoner_results.csv")

annotate("eedi_data", eedi_dataset, responses, results_df, "glm", "reasoning", "math")

## SciQ

In [ ]:
dump_random_traces("sciq_data", sciq_dataset, n_per_bucket=4)

### CoT

In [ ]:
with open("sciq_data/joint_results/deepseek-naive-cot-deepseek-chat_responses_by_datapointid.json", "r") as f:
    responses = json.load(f)
results_df = pd.read_csv("sciq_data/joint_results/deepseek-naive-cot-deepseek-chat_results.csv")

annotate("sciq_data", sciq_dataset, responses, results_df, "deepseek", "cot", "science", include_unsolvable=False)

In [ ]:
with open("sciq_data/joint_results/openrouter-naive-cot-z-ai_glm-4.7-chat_responses_by_datapointid.json", "r") as f:
    responses = json.load(f)
results_df = pd.read_csv("sciq_data/joint_results/openrouter-naive-cot-z-ai_glm-4.7-chat_results.csv")

annotate("sciq_data", sciq_dataset, responses, results_df, "glm", "cot", "science", include_unsolvable=False)

### Reasoning

In [ ]:
with open("sciq_data/joint_results/deepseek-naive-deepseek-reasoner_responses_by_datapointid.json", "r") as f:
    responses = json.load(f)
results_df = pd.read_csv("sciq_data/joint_results/deepseek-naive-deepseek-reasoner_results.csv")

annotate("sciq_data", sciq_dataset, responses, results_df, "deepseek", "reasoning", "science", include_unsolvable=False)

In [ ]:
with open("sciq_data/joint_results/openrouter-naive-z-ai_glm-4.7-reasoner_responses_by_datapointid.json", "r") as f:
    responses = json.load(f)
results_df = pd.read_csv("sciq_data/joint_results/openrouter-naive-z-ai_glm-4.7-reasoner_results.csv")

annotate("sciq_data", sciq_dataset, responses, results_df, "glm", "reasoning", "science", include_unsolvable=False)